# Variação da Sazonalidade do Fogo no Brasil Central
## Análise decadal do deslocamento da janela de queima (BWS) com verificação de robustez ENSO

**Projeto:** Fire Season — Brasil Central (Cerrado + Pantanal + zonas de transição)
**Dados:** MapBiomas Fire Coleção 4 (mensal) + LULC Coleção 9 + ONI/NOAA
**Período:** 1985–2024 | **Faixas latitudinais:** 5° de largura | **Classes:** vegetação natural × uso

---

### O que este notebook faz

Este notebook **consolida, refaz e documenta** a análise decadal do deslocamento do pico da
estação de fogo, que vinha distribuída em vários notebooks. Ele está organizado em 7 seções:

1. **Dados** — carga e descrição do CSV agregado (mês × faixa × tipo)
2. **Normalização** — taxa de queima por área disponível
3. **Métricas circulares** — BWS (pico) e Extensão (duração/dispersão) da estação
4. **Tendência (análise principal)** — Theil–Sen + Mann–Kendall sobre a série completa
5. **Verificações** — translação vs. alargamento; completude; delta entre pontas (ilustrativo)
6. **Robustez ENSO** — o deslocamento sobrevive ao controle pela fase ENSO?
7. **Saídas** — tabelas e figuras finais

> **Decisão metodológica central:** a medida oficial do deslocamento é a **tendência sobre os
> 40 anos** (Theil–Sen/Mann–Kendall), *não* a diferença entre duas décadas escolhidas. Isso evita
> o viés de selecionar o intervalo que dá significativo (um problema de comparações múltiplas).
> O delta entre décadas entra apenas como figura ilustrativa, com pontas fixas definidas a priori.


## 0 · Configuração do ambiente

Rode esta célula primeiro. No Google Colab ela instala as duas bibliotecas que podem faltar
(`pymannkendall` para o teste de tendência e, por garantia, `scipy`).

In [ ]:
# No Colab, descomente a linha abaixo na primeira execução:
# !pip -q install pymannkendall

import pandas as pd
import numpy as np
import math, re
import matplotlib.pyplot as plt
from scipy.stats import theilslopes, spearmanr
import pymannkendall as mk

# Constante para converter "meses" em "dias" (mês médio do calendário)
DIAS_POR_MES = 30.4

# Estética dos gráficos
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
print("Ambiente pronto.")

## 1 · Dados

O arquivo de entrada (`burn_monthly_by_latband_new.csv`) é o produto do geoprocessamento já
realizado (separação natural/uso pixel a pixel contra a LULC de cada ano, recorte na ROI do Brasil
Central, reprojeção para Albers EPSG:5880, tabulação por mês × faixa × tipo).

**Colunas:**

| coluna | significado |
|---|---|
| `ANO` | ano (1985–2024) |
| `MES` | mês (1–12) |
| `tipo_queima` | `nat` (vegetação natural) ou `uso` (uso antrópico) |
| `lat_band` | faixa latitudinal de 5°, ex. `[-15;-10]` |
| `Queima_km2` | área queimada no mês/faixa/tipo (km²) |
| `uso_km2` | área **disponível** de uso na faixa naquele ano (km²) — denominador |
| `nat_km2` | área **disponível** de vegetação natural na faixa naquele ano (km²) — denominador |

> Os denominadores `uso_km2`/`nat_km2` variam ano a ano (a área convertida cresce, a natural
> encolhe). Isso permite separar "queimou mais porque há mais área" de "queimou mais por unidade
> de área" — distinção essencial para não confundir expansão de fronteira com mudança de regime.

In [ ]:
# >>> AJUSTE O CAMINHO se necessário <<<
CSV_PATH = '/content/drive/MyDrive/Trabalho/Fire_Season/out_latbands/burn_monthly_by_latband_new.csv'
# Para teste local, descomente:
# CSV_PATH = 'burn_monthly_by_latband_new.csv'

df = pd.read_csv(CSV_PATH, sep=';')
df.columns = df.columns.str.strip().str.lower()
df = df.rename(columns={
    'ano':'year', 'mes':'month', 'tipo_queima':'tipo',
    'queima_km2':'burn_km2', 'uso_km2':'uso_km2', 'nat_km2':'nat_km2'
})
df['lat_band'] = df['lat_band'].str.replace(' ', '', regex=False)

print(f"Linhas: {len(df)}")
print(f"Período: {df.year.min()}–{df.year.max()}")
print(f"Faixas: {sorted(df.lat_band.unique())}")
print(f"Tipos: {list(df.tipo.unique())}")
df.head()

## 2 · Normalização por área disponível

A variável de análise não é a área queimada bruta, mas a **taxa de queima** — área queimada
dividida pela área disponível daquela classe na faixa/ano. Isso remove o efeito do tamanho da
faixa e da mudança de área convertida ao longo do tempo.

Trabalhamos com a área total disponível (`uso + nat`) como denominador comum, de modo que a taxa
seja comparável entre as duas classes.

In [ ]:
df['area_total_km2'] = df['uso_km2'] + df['nat_km2']
df = df[df['area_total_km2'] > 0].copy()      # remove faixas sem denominador válido
df['burn_rate'] = df['burn_km2'] / df['area_total_km2']

# latitude central de cada faixa (para ordenar e plotar)
def lat_mid(banda):
    a, b = re.findall(r'-?\d+', banda)
    return (int(a) + int(b)) / 2.0
df['lat_mid'] = df['lat_band'].map(lat_mid)

print("Taxa de queima calculada.")
print(df.groupby('tipo')['burn_rate'].describe()[['mean','min','max']])

## 3 · Métricas circulares: BWS e Extensão

A estação de fogo é um fenômeno **cíclico** — dezembro e janeiro são meses vizinhos, não opostos.
Por isso usamos **estatística circular**: cada mês vira um ângulo no círculo de 12 meses, e a
distribuição de queima ao longo do ano vira um conjunto de vetores.

Duas métricas resumem a estação de cada ano/faixa/tipo:

- **BWS (Burning Window Shift)** — o *mês médio ponderado* de queima, ou seja, a **posição do
  pico** da estação. É o ângulo médio dos vetores, convertido de volta para escala de mês (1–12).
  Um BWS maior = pico mais tarde no ano.

- **Extensão (1 − R)** — mede o quanto a queima está **espalhada** ao longo do ano. `R` é o
  comprimento do vetor resultante: R≈1 significa queima concentrada num único mês (estação curta);
  R≈0 significa queima espalhada (estação longa/difusa). Extensão = 1−R cresce com a dispersão.

> **Por que as duas juntas?** Um deslocamento do BWS pode significar duas coisas diferentes: o pico
> realmente migrou (translação), *ou* a estação ganhou uma cauda tardia que puxa a média sem o pico
> ter se movido (alargamento assimétrico). Olhar BWS **e** Extensão juntos distingue os dois casos —
> faremos isso na Seção 5.

In [ ]:
def metricas_circulares(pesos12):
    """Recebe 12 pesos mensais (taxa de queima jan..dez).
    Retorna (BWS, Extensao). BWS em escala de mês 1..12."""
    w = np.asarray(pesos12, dtype=float)
    total = w.sum()
    if total <= 0:
        return np.nan, np.nan
    theta = 2*np.pi*np.arange(12)/12.0          # ângulo de cada mês
    C = np.sum(w*np.cos(theta)) / total
    S = np.sum(w*np.sin(theta)) / total
    R = np.sqrt(C*C + S*S)                       # concentração (0..1)
    ang = math.atan2(S, C)
    if ang < 0:
        ang += 2*np.pi
    bws = (ang/(2*np.pi))*12.0 + 1.0             # volta para escala de mês
    if bws > 12:
        bws -= 12.0
    return bws, 1.0 - R                          # BWS, Extensão

# Calcular BWS e Extensão anuais por faixa × tipo
registros = []
for (ano, tipo, banda), sub in df.groupby(['year','tipo','lat_band']):
    pesos = np.zeros(12)
    for _, r in sub.iterrows():
        pesos[int(r['month'])-1] = r['burn_rate']
    bws, ext = metricas_circulares(pesos)
    registros.append(dict(year=ano, tipo=tipo, lat_band=banda,
                          lat_mid=lat_mid(banda), bws=bws, ext=ext))

anual = pd.DataFrame(registros).sort_values(['tipo','lat_mid','year']).reset_index(drop=True)
print(f"Métricas anuais: {anual.shape[0]} linhas (faixa × tipo × ano)")
print(f"BWS médio por tipo (mês):")
print(anual.groupby('tipo')['bws'].mean().round(2))
anual.head()

## 4 · Tendência do deslocamento — análise principal

Esta é a medida **oficial** do deslocamento. Em vez de comparar duas décadas, ajustamos a
tendência sobre a série inteira (1985–2024), por faixa × tipo, com dois métodos robustos:

- **Mann–Kendall** — testa se existe uma tendência monotônica (p-valor). Não assume reta perfeita
  e é resistente a valores extremos. É o padrão em climatologia/hidrologia.
- **Theil–Sen** — estima a inclinação (dias/década) como a *mediana* das inclinações entre todos
  os pares de pontos. Um El Niño anômalo não distorce a estimativa como faria numa regressão comum.

> **Por que robusto importa aqui:** os anos de El Niño são exatamente os "outliers" que poderiam
> puxar uma regressão comum. Usar Theil–Sen já reduz a influência deles *antes* mesmo da
> verificação ENSO formal da Seção 6.

In [ ]:
def tendencia(serie_anos, serie_valores):
    """Theil-Sen (dias/década) + Mann-Kendall (p, direção)."""
    x = np.asarray(serie_anos, float)
    y = np.asarray(serie_valores, float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(y) < 5:
        return np.nan, np.nan, 'insuficiente'
    slope, intercept, lo, hi = theilslopes(y, x)   # slope em mês/ano
    dias_decada = slope * 10 * DIAS_POR_MES
    r = mk.original_test(y)
    return dias_decada, r.p, r.trend

linhas = []
for tipo in ['nat','uso']:
    for banda in sorted(anual.lat_band.unique(), key=lat_mid):
        s = anual[(anual.tipo==tipo) & (anual.lat_band==banda)].sort_values('year')
        d_dec, p, tend = tendencia(s.year, s.bws)
        linhas.append(dict(tipo=tipo, lat_band=banda, lat_mid=lat_mid(banda),
                           dias_por_decada=d_dec, MK_p=p, tendencia=tend,
                           significativo=(p < 0.05) if np.isfinite(p) else False))
tend_bws = pd.DataFrame(linhas)
print("TENDÊNCIA DO BWS (deslocamento do pico da estação de fogo)\n")
print(tend_bws.to_string(index=False))

**Como ler a tabela acima:**

- `dias_por_decada` positivo = o pico da estação se desloca para **mais tarde** no ano.
- `significativo` = True quando Mann–Kendall dá p < 0,05.
- Espera-se o sinal concentrado na **vegetação natural** das faixas centrais, e ausente no **uso**
  (que segue calendário antrópico, não clima).

## 5 · Verificações

### 5.1 · Translação ou alargamento? (BWS × Extensão)

Se o BWS aumenta **e** a Extensão também aumenta, o "deslocamento" do pico é em parte um
**alargamento assimétrico** (a estação ganhou cauda tardia). Se o BWS aumenta mas a Extensão fica
estável, é **translação pura** (a estação inteira se moveu). Testamos a tendência da Extensão com
o mesmo método.

In [ ]:
linhas_ext = []
for tipo in ['nat','uso']:
    for banda in sorted(anual.lat_band.unique(), key=lat_mid):
        s = anual[(anual.tipo==tipo) & (anual.lat_band==banda)].sort_values('year')
        d_dec, p, tend = tendencia(s.year, s.ext)
        linhas_ext.append(dict(tipo=tipo, lat_band=banda, lat_mid=lat_mid(banda),
                               ext_tend_por_decada=d_dec/DIAS_POR_MES/10,  # de volta p/ unidade 1-R
                               MK_p=p, tendencia=tend))
tend_ext = pd.DataFrame(linhas_ext)

# juntar BWS e Extensão lado a lado para vegetação natural
comp = tend_bws[tend_bws.tipo=='nat'][['lat_band','lat_mid','dias_por_decada','MK_p','significativo']]
comp = comp.rename(columns={'dias_por_decada':'BWS_dias_dec','MK_p':'BWS_p','significativo':'BWS_signif'})
comp = comp.merge(
    tend_ext[tend_ext.tipo=='nat'][['lat_band','MK_p','tendencia']].rename(
        columns={'MK_p':'Ext_p','tendencia':'Ext_tend'}), on='lat_band')
print("VEGETAÇÃO NATURAL — BWS vs Extensão\n")
print(comp.sort_values('lat_mid').to_string(index=False))
print("\nLeitura: se Ext_tend = 'no trend' onde BWS é significativo,")
print("o deslocamento é TRANSLAÇÃO (pico se move), não alargamento.")

### 5.2 · Completude dos dados

Confere se todas as combinações ano × faixa × tipo estão presentes — buracos enviesam tendências.

In [ ]:
esperado = (anual.year.nunique() * anual.tipo.nunique() * anual.lat_band.nunique())
print(f"Combinações esperadas: {esperado} | presentes: {len(anual)}")
faltantes = esperado - len(anual)
print("✅ Dados completos." if faltantes == 0 else f"⚠ {faltantes} combinações ausentes.")

# BWS válido (não-NaN) por tipo e faixa
print("\nAnos com BWS válido por tipo × faixa:")
print(anual.dropna(subset=['bws']).groupby(['tipo','lat_band']).size().unstack('tipo'))

### 5.3 · Delta entre pontas (apenas ilustrativo)

Comparação **com pontas fixadas a priori**: primeira década (1985–1994) vs. última (2015–2024).
Isto **não é** a medida oficial (Seção 4) — serve só para visualização. As pontas são fixas para
evitar o viés de escolher o intervalo significativo.

In [ ]:
BASE = (1985, 1994)   # primeira década — fixada a priori
RECENTE = (2015, 2024) # última década — fixada a priori

def media_decada(sub, ini, fim):
    return sub[(sub.year>=ini)&(sub.year<=fim)]['bws'].mean()

delta_rows = []
for tipo in ['nat','uso']:
    for banda in sorted(anual.lat_band.unique(), key=lat_mid):
        s = anual[(anual.tipo==tipo)&(anual.lat_band==banda)]
        b = media_decada(s, *BASE); r = media_decada(s, *RECENTE)
        delta_rows.append(dict(tipo=tipo, lat_band=banda, lat_mid=lat_mid(banda),
                               dBWS_meses=r-b, dBWS_dias=(r-b)*DIAS_POR_MES))
delta = pd.DataFrame(delta_rows)
print(f"ΔBWS entre {BASE[0]}–{BASE[1]} e {RECENTE[0]}–{RECENTE[1]} (ilustrativo)\n")
print(delta[delta.tipo=='nat'].sort_values('lat_mid').to_string(index=False))

## 6 · Robustez ENSO

**A pergunta crítica:** o deslocamento do pico (Seção 4) é uma tendência climática secular, ou é
artefato dos El Niños estarem concentrados nas décadas recentes? Se anos de El Niño têm pico
naturalmente mais tarde, parte da tendência poderia ser apenas a composição da amostra.

Testamos isso em duas etapas, **por trimestre do ONI separadamente** (não pela média), porque o
estado ENSO evolui dentro da estação seca e queremos saber *qual momento* importa.

A estação de fogo do Brasil Central concentra-se em **JJA, JAS, ASO, SON** (≈80% da seca).
Tratamos cada um como classificador independente.

In [ ]:
# ---- Carregar e estruturar o ONI ----
# >>> AJUSTE O CAMINHO <<<
ONI_PATH = '/content/drive/MyDrive/Trabalho/Fire_Season/ElNino.txt'
# ONI_PATH = 'ElNino.txt'  # teste local

raw = open(ONI_PATH).read()
oni_rows = []
for line in raw.splitlines():
    m = re.match(r'^(19\d\d|20\d\d)\s+(.*)$', line.strip())
    if m:
        vals = m.group(2).split()
        oni_rows.append([int(m.group(1))] + [float(v) for v in vals])
oni_cols = ['year','DJF','JFM','FMA','MAM','AMJ','MJJ','JJA','JAS','ASO','SON','OND','NDJ']
oni = pd.DataFrame([r + [np.nan]*(13-len(r)) for r in oni_rows], columns=oni_cols)
print("ONI carregado:", oni.year.min(), "–", oni.year.max())
oni[['year','JJA','JAS','ASO','SON']].tail()

### 6.1 · Correlação BWS × ONI por trimestre (vegetação natural)

Para cada faixa, correlacionamos a série de BWS com cada trimestre do ONI (Spearman, robusto a
não-linearidade). Procuramos: **onde há tendência, há também correlação com o ONI?** Se não houver,
a tendência é independente do ENSO.

In [ ]:
print("Correlação de Spearman — BWS × ONI trimestral (Veg. Natural)")
print("="*72)
hdr = f"{'faixa':12}" + "".join(f"{q:>14}" for q in ['JJA','JAS','ASO','SON'])
print(hdr)
for banda in sorted(anual.lat_band.unique(), key=lat_mid):
    s = (anual[(anual.tipo=='nat')&(anual.lat_band==banda)]
         .merge(oni[['year','JJA','JAS','ASO','SON']], on='year'))
    linha = f"{banda:12}"
    for q in ['JJA','JAS','ASO','SON']:
        rho, p = spearmanr(s.bws, s[q])
        estrela = '*' if p < 0.05 else ' '
        linha += f"  r={rho:+.2f} p={p:.2f}{estrela}"
    print(linha)
print("\n* = p < 0.05. Faixas SEM estrela onde há tendência = sinal independente do ENSO.")

### 6.2 · A tendência sobrevive ao controle ENSO?

Teste decisivo: removemos o efeito do ONI do BWS (regressão linear) e testamos se a tendência
**do resíduo** continua significativa. Se o p-valor residual continuar baixo, a tendência não
depende do ENSO — está blindada.

In [ ]:
print("Tendência do BWS antes e depois de remover o efeito ONI (Veg. Natural)")
print("="*72)
print(f"{'faixa':12} {'MK p (bruto)':>14} {'MK p (resíduo)':>16}  {'trimestre ONI'}")
TRIM = 'ASO'   # trimestre de controle (pico da estação); pode trocar por SON/JAS/JJA
for banda in sorted(anual.lat_band.unique(), key=lat_mid):
    s = (anual[(anual.tipo=='nat')&(anual.lat_band==banda)]
         .merge(oni[['year',TRIM]], on='year').sort_values('year'))
    p_bruto = mk.original_test(s.bws.values).p
    # remover ONI por regressão linear e testar tendência do resíduo
    coef = np.polyfit(s[TRIM].values, s.bws.values, 1)
    resid = s.bws.values - np.polyval(coef, s[TRIM].values)
    p_resid = mk.original_test(resid).p
    print(f"{banda:12} {p_bruto:14.3f} {p_resid:16.3f}  {TRIM}")
print("\nSe 'MK p (resíduo)' continua < 0.05 onde 'MK p (bruto)' era < 0.05,")
print("o deslocamento do pico é ROBUSTO ao ENSO — tendência secular genuína.")

## 7 · Saídas e figuras

### 7.1 · Figura principal — tendência por latitude

In [ ]:
fig, ax = plt.subplots(figsize=(8,5.5))
for tipo, cor, mk_ in [('nat','#c0392b','o'), ('uso','#2471a3','s')]:
    sub = tend_bws[tend_bws.tipo==tipo].sort_values('lat_mid')
    signif = sub.significativo.fillna(False)
    ax.plot(sub.lat_mid, sub.dias_por_decada, '-', color=cor, alpha=0.4, zorder=1)
    ax.scatter(sub.lat_mid[signif], sub.dias_por_decada[signif],
               color=cor, marker=mk_, s=90, zorder=3,
               label=f'{tipo} (significativo)', edgecolor='k', linewidth=0.5)
    ax.scatter(sub.lat_mid[~signif], sub.dias_por_decada[~signif],
               facecolor='white', edgecolor=cor, marker=mk_, s=90, zorder=2,
               label=f'{tipo} (n.s.)')
ax.axhline(0, color='k', lw=0.8)
ax.set_xlabel('Latitude central da faixa (°)')
ax.set_ylabel('Deslocamento do pico (dias/década)')
ax.set_title('Deslocamento da estação de fogo por latitude\nBrasil Central, 1985–2024')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('fig_tendencia_bws_latitude.png', dpi=160, bbox_inches='tight')
plt.show()
print("Figura salva: fig_tendencia_bws_latitude.png")

### 7.2 · Exportar tabelas de resultado

In [ ]:
# Tabela mestre de tendências (BWS) + extensão + delta ilustrativo
tend_bws.to_csv('resultado_tendencia_BWS.csv', index=False)
tend_ext.to_csv('resultado_tendencia_Extensao.csv', index=False)
anual.to_csv('metricas_anuais_BWS_por_faixa_tipo.csv', index=False)
delta.to_csv('delta_decadal_ilustrativo.csv', index=False)
print("Arquivos salvos:")
for f in ['resultado_tendencia_BWS.csv','resultado_tendencia_Extensao.csv',
          'metricas_anuais_BWS_por_faixa_tipo.csv','delta_decadal_ilustrativo.csv']:
    print("  -", f)

---

## Resumo metodológico (para o paper)

1. **Unidade de análise:** BWS anual (mês médio circular ponderado pela taxa de queima) por faixa
   latitudinal de 5° × tipo (natural/uso), 1985–2024, dentro da ROI do Brasil Central.

2. **Normalização:** taxa de queima = área queimada / área disponível da classe (ano a ano),
   separando mudança de fronteira de mudança de regime.

3. **Medida do deslocamento:** tendência de Theil–Sen (dias/década) com significância por
   Mann–Kendall, sobre a série completa — **não** delta entre décadas selecionadas (evita
   comparações múltiplas). Delta entre pontas fixas (1985–94 vs 2015–24) só como ilustração.

4. **Distinção translação/alargamento:** tendência conjunta de BWS e Extensão (1−R).

5. **Robustez ENSO:** correlação BWS × ONI por trimestre individual (JJA/JAS/ASO/SON) e teste de
   tendência do resíduo após remoção do efeito ONI. O deslocamento é considerado secular se
   sobrevive ao controle.

> **Achado-chave esperado:** deslocamento tardio significativo na vegetação natural das faixas
> centrais, ausente no uso, **independente do ENSO** — com o ENSO modulando apenas a faixa norte
> (arco amazônico), onde não há tendência. Dissociação espacial entre tendência secular e
> variabilidade ENSO.
